# Radix 2 and 2^2 FFT

In [1]:
import numpy as np
import fxpmath as fxp
from scipy.fft import fft, ifft

In [2]:
class radix2_PreAdder:
    def __init__(self):
        self.input_a = 0.0
        self.input_b = 0.0
        self.output_add = 0.0
        self.output_sub = 0.0


    def calculate(self):
        self.output_add = self.input_a + self.input_b
        self.output_sub = self.input_a - self.input_b
        

class radix2_Rotator:
    cnt = 0
    def __init__(self, stage_index, num_of_stages):
        self.input = 0.0
        self.output = 0.0
        self.stage_index = stage_index
        self.twiddleROM = (np.ones(2**(num_of_stages-stage_index))).astype(complex)
        self.half_len = 2**(num_of_stages-stage_index-1)
        N = 2**num_of_stages
        for i in range(self.half_len):
            k = i * 2**(stage_index)
            self.twiddleROM[i+self.half_len] = np.exp(-1j*2*np.pi*k/N)
        
        print(f"STAGE {stage_index}, twiddle = {self.twiddleROM}")

        # print(self.twiddleROM)

    def rotate(self, fifo_full_flag):
        if (fifo_full_flag): # dozvola da brojac vrti i cita redom twiddle faktore iz memorije
            print(f"STAGE {self.stage_index}, FIFO FULL")
            self.output = self.input * self.twiddleROM[self.cnt]
            print(f'STAGE {self.stage_index}, curr twiddle = {self.twiddleROM[self.cnt]}')
            if (self.cnt == self.half_len*2-1):
                self.cnt = 0
            else:
                self.cnt += 1
        

class Fifo:
    full = 0
    cnt = 0

    def __init__(self, depth):
        self.depth = depth
        self.buffer = (np.zeros(depth)).astype(complex)
        # print(self.depth)

    def is_full(self):
        return self.full
    
    def get_output(self):
        return self.buffer[-1]
    
    def shift(self, input_sample):
        self.cnt += 1
        self.buffer = np.roll(self.buffer, 1)
        self.buffer[0] = input_sample
        if (self.cnt > self.depth):
            self.full = 1
        else:
            self.full = 0



In [3]:
class radix2_SDF_stage:
    input_sample = 0.0
    output_sample = 0.0
    op_cnt = 0 # operation counter (counting how many add/subb operations are done)
    def __init__(self, stage_index, num_of_stages):
        self.stage_index = stage_index
        self.num_of_stages = num_of_stages
        self.num_of_samples = 2**(num_of_stages-stage_index)
        self.fifo = Fifo(2**(num_of_stages-stage_index-1))
        self.pre_adder = radix2_PreAdder()
        self.rotator = radix2_Rotator(stage_index=stage_index, num_of_stages=num_of_stages)

    def isFifoFull(self):
        return self.fifo.is_full()
    
    def calculate(self):
        print(f'STAGE {self.stage_index}, op_cnt = ', self.op_cnt)
        self.pre_adder.input_a = self.fifo.get_output()
        self.pre_adder.input_b = self.input_sample
        self.pre_adder.calculate()

        print(f'STAGE {self.stage_index}, add_out = {self.pre_adder.output_add}')
        print(f'STAGE {self.stage_index}, sub_out = {self.pre_adder.output_sub}')
        
        if (self.op_cnt//(self.num_of_samples/2)): ## other half of the input stream is comming
            self.output_sample = self.pre_adder.output_add
            self.fifo.shift(self.pre_adder.output_sub) 
        else:
            self.output_sample = self.fifo.get_output()
            self.fifo.shift(self.input_sample)

        self.rotator.input = self.output_sample
        self.rotator.rotate(self.isFifoFull())
        self.output_sample = self.rotator.output

        # print(f'stage_{self.stage_index} : {self.output_sample}')

        if (self.op_cnt == self.num_of_samples-1):
            self.op_cnt = 0
        else:
            self.op_cnt += 1

        print(f'STAGE {self.stage_index}, output = {self.output_sample}')


In [4]:
stage0 = radix2_SDF_stage(stage_index=0, num_of_stages=3)
stage1 = radix2_SDF_stage(stage_index=1, num_of_stages=3)
stage2 = radix2_SDF_stage(stage_index=2, num_of_stages=3)


# input_vector = [1.0, 2.0, 3.0, 4.0, 0.0, 0.0, 0.0]
# input_vector = [4.0, 3.0, 2.0, 1.0, 0.0, 0.0, 0.0, 0.0]

# input_vector = [1.0, 1.0, 0.0]
input_vector = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
fft_manual = []
for i in range(len(input_vector)):
    stage0.input_sample = input_vector[i]
    stage0.calculate()
    stage1.input_sample = stage0.output_sample
    stage1.calculate()
    stage2.input_sample = stage1.output_sample
    stage2.calculate()
    print(stage2.output_sample)
    if i > 6:
        fft_manual.append(stage2.output_sample)


STAGE 0, twiddle = [ 1.00000000e+00+0.j          1.00000000e+00+0.j
  1.00000000e+00+0.j          1.00000000e+00+0.j
  1.00000000e+00+0.j          7.07106781e-01-0.70710678j
  6.12323400e-17-1.j         -7.07106781e-01-0.70710678j]
STAGE 1, twiddle = [1.000000e+00+0.j 1.000000e+00+0.j 1.000000e+00+0.j 6.123234e-17-1.j]
STAGE 2, twiddle = [1.+0.j 1.+0.j]
STAGE 0, op_cnt =  0
STAGE 0, add_out = (1+0j)
STAGE 0, sub_out = (-1+0j)
STAGE 0, output = 0.0
STAGE 1, op_cnt =  0
STAGE 1, add_out = 0j
STAGE 1, sub_out = 0j
STAGE 1, output = 0.0
STAGE 2, op_cnt =  0
STAGE 2, add_out = 0j
STAGE 2, sub_out = 0j
STAGE 2, output = 0.0
0.0
STAGE 0, op_cnt =  1
STAGE 0, add_out = (2+0j)
STAGE 0, sub_out = (-2+0j)
STAGE 0, output = 0.0
STAGE 1, op_cnt =  1
STAGE 1, add_out = 0j
STAGE 1, sub_out = 0j
STAGE 1, output = 0.0
STAGE 2, op_cnt =  1
STAGE 2, add_out = 0j
STAGE 2, sub_out = 0j
STAGE 2, FIFO FULL
STAGE 2, curr twiddle = (1+0j)
STAGE 2, output = 0j
0j
STAGE 0, op_cnt =  2
STAGE 0, add_out = (3+0j)
S

In [5]:
def bracewell_buneman(xarray, length, log2length):
    ''' 
    bracewell-buneman bit reversal function
    inputs: xarray is array; length is array length; log2length=log2(length).
    output: bit reversed array xarray. 
    '''
    muplus = int((log2length+1)/2)
    mvar = 1
    reverse = np.zeros(length, dtype = int)
    upper_range = muplus+1
    for _ in np.arange(1, upper_range):
        for kvar in np.arange(0, mvar):
            tvar = 2*reverse[kvar]
            reverse[kvar] = tvar
            reverse[kvar+mvar] = tvar+1
        mvar = mvar+mvar
    if (log2length & 0x01):
            mvar = mvar/2

    mvar = int(mvar)
    for qvar in np.arange(1, mvar):
        
        nprime = qvar-mvar
        rprimeprime = reverse[qvar]*mvar
        for pvar in np.arange(0, reverse[qvar]):
            nprime = nprime+mvar
            rprime = rprimeprime+reverse[pvar]
            temp = xarray[nprime]
            xarray[nprime] = xarray[rprime]
            xarray[rprime] = temp
    return xarray

In [6]:
# print(fft_manual)
fft_manual = np.array(bracewell_buneman(fft_manual, len(fft_manual), int(np.log2(len(fft_manual)))))
print(fft_manual)

[36.+0.j         -4.+9.65685425j -4.+4.j         -4.+1.65685425j
 -4.+0.j         -4.-1.65685425j -4.-4.j         -4.-9.65685425j]


In [7]:
# input_vector = [1.0, 1.0]
input_vector = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]
# input_vector = [1.0, 2.0, 3.0, 4.0]
fft_numpy = np.fft.fft(input_vector)
print(fft_numpy)

[36.+0.j         -4.+9.65685425j -4.+4.j         -4.+1.65685425j
 -4.+0.j         -4.-1.65685425j -4.-4.j         -4.-9.65685425j]


In [8]:
print(np.allclose(fft_manual,fft_numpy))

True


## Radix-3 SDF Butterfly 

In [9]:
class radix3_PreAdder:
    def __init__(self):
        self.input_0 = 0.0
        self.input_1 = 0.0
        self.input_2 = 0.0
        self.output_0 = 0.0
        self.output_1 = 0.0
        self.output_2 = 0.0


    def calculate(self):
        tmp_0_0 = self.input_0
        tmp_1_0 = self.input_1 + self.input_2
        tmp_2_0 = self.input_1 - self.input_2
        ###### prvi nivo pajplajna ^
        tmp_0_1 = tmp_0_0 + tmp_1_0
        tmp_1_1 = tmp_0_0 - (1/2)*tmp_1_0
        tmp_2_1 = tmp_2_0 * (-1j*np.sqrt(3)/2)
        ###### drugi nivo pajplajna ^
        tmp_0_2 = tmp_0_1
        tmp_1_2 = tmp_1_1 + tmp_2_1
        tmp_2_2 = tmp_1_1 - tmp_2_1
        ###### treci nivo pajplajna ^ (ovo su izlazni registri vrv)
        self.output_0 = tmp_0_2
        self.output_1 = tmp_1_2
        self.output_2 = tmp_2_2

class radix3_Rotator:
    cnt = 0
    def __init__(self, stage_index, num_of_stages):
        self.input = 0.0
        self.output = 0.0
        self.stage_index = stage_index
        self.twiddleROM = (np.ones(3**(num_of_stages-stage_index))).astype(complex)
        self.two_thirds_len = 3**(num_of_stages-stage_index) - (3**(num_of_stages-stage_index)//3)
        # print(self.two_thirds_len)
        N = 3**num_of_stages
        if (self.two_thirds_len > 2):
            for i in range(self.two_thirds_len):
                if (i < self.two_thirds_len//2):
                    k = i * 3**(stage_index)
                else:
                    k = 2*(i-self.two_thirds_len//2) * 3**(stage_index)
                # print("k = ", k)
                self.twiddleROM[i+(len(self.twiddleROM) - self.two_thirds_len)] = np.exp(-1j*2*np.pi*k/N)
        
        print(f"STAGE {stage_index}, twiddle = {self.twiddleROM}")

        # print(self.twiddleROM)

    def rotate(self, fifo_full_flag):
        if (fifo_full_flag): # dozvola da brojac vrti i cita redom twiddle faktore iz memorije
            # print(f"STAGE {self.stage_index}, FIFO FULL")
            self.output = self.input * self.twiddleROM[self.cnt]
            # print(f'STAGE {self.stage_index}, curr twiddle = {self.twiddleROM[self.cnt]}')
            if (self.cnt == len(self.twiddleROM)-1):
                self.cnt = 0
            else:
                self.cnt += 1

In [10]:
class radix3_SDF_stage:
    input_sample = 0.0
    output_sample = 0.0
    op_cnt = 0 # operation counter (counting how many add/subb operations are done)
    def __init__(self, stage_index, num_of_stages):
        self.stage_index = stage_index
        self.num_of_stages = num_of_stages
        self.num_of_samples = 3**(num_of_stages-stage_index)
        self.fifo_0 = Fifo(3**(num_of_stages-stage_index-1))
        self.fifo_1 = Fifo(3**(num_of_stages-stage_index-1))
        self.pre_adder = radix3_PreAdder()
        self.rotator = radix3_Rotator(stage_index=stage_index, num_of_stages=num_of_stages)

    def isFifoFull_0(self):
        return self.fifo_0.is_full()
    def isFifoFull_1(self):
        return self.fifo_1.is_full()
    
    def calculate(self):
        # print(f'STAGE {self.stage_index}, op_cnt = ', self.op_cnt)
        self.pre_adder.input_0 = self.fifo_0.get_output()
        self.pre_adder.input_1 = self.fifo_1.get_output()
        self.pre_adder.input_2 = self.input_sample
        self.pre_adder.calculate()

        # print(f'STAGE {self.stage_index}, pre_adder_out_0 = {self.pre_adder.output_0}')
        # print(f'STAGE {self.stage_index}, pre_adder_out_1 = {self.pre_adder.output_1}')
        # print(f'STAGE {self.stage_index}, pre_adder_out_2 = {self.pre_adder.output_2}')
        
        if (self.op_cnt < (self.num_of_samples//3)): ## other half of the input stream is comming
            self.output_sample = self.fifo_0.get_output()
            self.fifo_0.shift(self.input_sample)
        elif ((self.op_cnt >= (self.num_of_samples//3)) and (self.op_cnt < (self.num_of_samples*2/3))):
            self.output_sample = self.fifo_1.get_output()
            self.fifo_1.shift(self.input_sample)
        else:
            self.output_sample = self.pre_adder.output_0
            self.fifo_0.shift(self.pre_adder.output_1)
            self.fifo_1.shift(self.pre_adder.output_2)

        self.rotator.input = self.output_sample
        self.rotator.rotate(self.isFifoFull_1())
        self.output_sample = self.rotator.output

        # print(f'stage_{self.stage_index} : {self.output_sample}')

        if (self.op_cnt == self.num_of_samples-1):
            self.op_cnt = 0
        else:
            self.op_cnt += 1

        # print(f'STAGE {self.stage_index}, output = {self.output_sample}')

In [11]:
stage0_radix3 = radix3_SDF_stage(stage_index=0, num_of_stages=3)
stage1_radix3 = radix3_SDF_stage(stage_index=1, num_of_stages=3)
stage2_radix3 = radix3_SDF_stage(stage_index=2, num_of_stages=3)

input_vector = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
input_vector = np.random.random(27)
input_vector_padded = np.append(input_vector, np.zeros(26))
fft_radix3_manual = []
for i in range(len(input_vector_padded)):
    stage0_radix3.input_sample = input_vector_padded[i]
    stage0_radix3.calculate()
    stage1_radix3.input_sample = stage0_radix3.output_sample
    stage1_radix3.calculate()
    stage2_radix3.input_sample = stage1_radix3.output_sample
    stage2_radix3.calculate()
    print(stage2_radix3.output_sample)
    if i >= 26:
        fft_radix3_manual.append(stage2_radix3.output_sample)

fft_radix3_manual = np.array(fft_radix3_manual)

STAGE 0, twiddle = [ 1.        +0.j          1.        +0.j          1.        +0.j
  1.        +0.j          1.        +0.j          1.        +0.j
  1.        +0.j          1.        +0.j          1.        +0.j
  1.        +0.j          0.97304487-0.23061587j  0.89363264-0.44879918j
  0.76604444-0.64278761j  0.59715859-0.80212319j  0.39607977-0.91821611j
  0.17364818-0.98480775j -0.05814483-0.99830816j -0.28680323-0.95798951j
  1.        +0.j          0.89363264-0.44879918j  0.59715859-0.80212319j
  0.17364818-0.98480775j -0.28680323-0.95798951j -0.68624164-0.72737364j
 -0.93969262-0.34202014j -0.99323836+0.11609291j -0.83548781+0.54950898j]
STAGE 1, twiddle = [ 1.        +0.j          1.        +0.j          1.        +0.j
  1.        +0.j          0.76604444-0.64278761j  0.17364818-0.98480775j
  1.        +0.j          0.17364818-0.98480775j -0.93969262-0.34202014j]
STAGE 2, twiddle = [1.+0.j 1.+0.j 1.+0.j]
0.0
0.0
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j


In [12]:
import numpy as np

def digit_reverse_array(arr, radix):
    """
    Perform digit-reversal on a NumPy array for a given radix.
    
    Parameters:
        arr (np.ndarray): Input array to be reordered.
        radix (int): The radix (base) for the FFT.
    
    Returns:
        np.ndarray: Reordered array based on digit-reversal indices.
    """
    n = arr.size
    if not np.log(n) / np.log(radix) % 1 == 0:
        raise ValueError("The size of the array must be a power of the radix.")
    
    num_digits = int(np.log(n) / np.log(radix))
    
    def digit_reverse(index, radix, num_digits):
        reversed_index = 0
        for _ in range(num_digits):
            reversed_index = reversed_index * radix + (index % radix)
            index //= radix
        return reversed_index
    
    reordered = np.empty_like(arr)
    for i in range(n):
        reversed_index = digit_reverse(i, radix, num_digits)
        reordered[reversed_index] = arr[i]
    
    return reordered


In [13]:
# print(fft_radix3_manual)
fft_radix3_manual = digit_reverse_array(fft_radix3_manual, radix=3)
for num in fft_radix3_manual:
    print(num)
# fft_radix3_numpy = np.fft.fft([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0])
fft_radix3_numpy = np.fft.fft(input_vector)
print("\n")
for num in fft_radix3_numpy:
    print(num)
# print(fft_radix3_numpy)

(13.566915431535193+0j)
(-1.3113060421374665+0.32656667960925473j)
(1.856192467941842+1.77939693457524j)
(0.12780654188477358-0.3486625783968674j)
(1.297707664071019-0.33771000745838736j)
(-0.15558054217342343+0.7656919931865322j)
(-1.0808384029300944+1.5604808617991583j)
(1.0659126454046253-0.5098380711956874j)
(-0.259097841944255+1.6206024820602494j)
(-1.205617584229497+1.24514204458611j)
(-1.825149784545156+0.4372300323472732j)
(0.20418790793941033-0.162461190611638j)
(1.3217026391186484+0.420656984235919j)
(1.061198177267545-0.8389960537123979j)
(1.0611981772675445+0.8389960537123984j)
(1.3217026391186484-0.4206569842359188j)
(0.2041879079394101+0.16246119061163788j)
(-1.825149784545156-0.43723003234727403j)
(-1.205617584229497-1.24514204458611j)
(-0.2590978419442548-1.6206024820602494j)
(1.0659126454046253+0.5098380711956878j)
(-1.0808384029300941-1.5604808617991583j)
(-0.15558054217342315-0.7656919931865322j)
(1.2977076640710188+0.3377100074583874j)
(0.12780654188477358+0.3486625

In [14]:
twiddles = np.ones(9).astype(complex)
for i in range(6):
    k = i * 3**(0)
    print(k)
    twiddles[i+(9 - 6)] = np.exp(-1j*2*np.pi*k/9)

print(twiddles)


0
1
2
3
4
5
[ 1.        +0.j          1.        +0.j          1.        +0.j
  1.        +0.j          0.76604444-0.64278761j  0.17364818-0.98480775j
 -0.5       -0.8660254j  -0.93969262-0.34202014j -0.93969262+0.34202014j]


## Radix-5 SDF Butterfly 

In [ ]:
class radix5_PreAdder:
    def __init__(self):
        self.input_0 = 0.0
        self.input_1 = 0.0
        self.input_2 = 0.0
        self.input_3 = 0.0
        self.input_4 = 0.0
        self.output_0 = 0.0
        self.output_1 = 0.0
        self.output_2 = 0.0
        self.output_3 = 0.0
        self.output_4 = 0.0


    def calculate(self):
        tmp_0_0 = self.input_0
        tmp_1_0 = self.input_1 + self.input_4
        tmp_2_0 = self.input_2 + self.input_3
        tmp_3_0 = self.input_1 - self.input_4
        tmp_4_0 = self.input_2 - self.input_3
        ###### prvi nivo pajplajna ^
        tmp_0_1 = tmp_0_0
        tmp_1_1 = tmp_1_0 + tmp_2_0
        tmp_2_1 = tmp_1_0 - tmp_2_0
        tmp_3_1 = tmp_3_0
        tmp_4_1 = tmp_4_0
        tmp_5_1 = tmp_3_0 + tmp_4_0 # dodatna grana izmedju
        ###### drugi nivo pajplajna ^
        tmp_0_2 = tmp_0_1 + tmp_1_1
        tmp_1_2 = tmp_0_1 + tmp_1_1 * (-0.25)
        tmp_2_2 = tmp_1_1 * 0.559
        tmp_3_2 = tmp_3_1 * (-1j*0.363)
        tmp_4_2 = tmp_4_1 * 1j*1.539
        tmp_5_2 = tmp_5_1 * (-1j*0.588)
        ###### treci nivo pajplajna ^
        tmp_0_3 = tmp_0_2
        tmp_1_3 = tmp_1_2 + tmp_2_2
        tmp_2_3 = tmp_1_2 - tmp_2_2
        tmp_3_3 = tmp_3_2 + tmp_5_2
        tmp_4_3 = tmp_4_2 + tmp_5_2
        ###### cetvrti nivo pajplajna ^
        self.output_0 = tmp_0_3
        self.output_1 = tmp_1_3 + tmp_3_3
        self.output_2 = tmp_2_3 + tmp_4_3
        self.output_4 = tmp_1_3 - tmp_3_3
        self.output_3 = tmp_2_3 - tmp_4_3

class radix5_Rotator:
    cnt = 0
    def __init__(self, stage_index, num_of_stages):
        self.input = 0.0
        self.output = 0.0
        self.stage_index = stage_index
        self.twiddleROM = (np.ones(5**(num_of_stages-stage_index))).astype(complex)
        self.four_fifths_len = 5**(num_of_stages-stage_index) - (5**(num_of_stages-stage_index)//5)
        N = 5**num_of_stages
        if (self.four_fifths_len > 4):
            for i in range(self.four_fifths_len):
                if (i < self.four_fifths_len//2):
                    k = i * 5**(stage_index)
                else:
                    # UPITNO
                    k = 2*(i-self.two_thirds_len//2) * 3**(stage_index)
                # print("k = ", k)
                self.twiddleROM[i+(len(self.twiddleROM) - self.four_fifths_len)] = np.exp(-1j*2*np.pi*k/N)
        
        print(f"STAGE {stage_index}, twiddle = {self.twiddleROM}")

        # print(self.twiddleROM)

    def rotate(self, fifo_full_flag):
        if (fifo_full_flag): # dozvola da brojac vrti i cita redom twiddle faktore iz memorije
            # print(f"STAGE {self.stage_index}, FIFO FULL")
            self.output = self.input * self.twiddleROM[self.cnt]
            # print(f'STAGE {self.stage_index}, curr twiddle = {self.twiddleROM[self.cnt]}')
            if (self.cnt == len(self.twiddleROM)-1):
                self.cnt = 0
            else:
                self.cnt += 1

## Decompositon of N on $2^i \cdot 3^j \cdot 5^k$

The max number of subcarriers in OFDM (5G NR standard) is 3300 $(275 * 12)$, so besides radix 2, 3 and 5, a radix 11 butterfly ($275 = 5^2 \cdot 11^1$) is also needed to achieve the best performance (lowest spectral leakage).
In the next few cells radix powers and the list of all possible FFT sizes will be generated.

It is also possible to avoid the radix 11 butterfly usage if the system is willing to introduce some error because of the spectral leakage. In that case, radices 2, 3 and 5 could generate FFT of size 3840.

In [15]:
i_arr = []
j_arr = []
k_arr = []
l_arr = []
for i in range(10):
    for j in range(10):
        for k in range(10):
            for l in range(10):
                if ((((2**i) * (3**j) * (5**k) * (11**l)) <= 275)):
                    i_arr.append(i)
                    j_arr.append(j)
                    k_arr.append(k)
                    l_arr.append(l)

In [16]:
N_arr = []
i_arr = np.unique(i_arr)
j_arr = np.unique(j_arr)
k_arr = np.unique(k_arr)
l_arr = np.unique(l_arr)

for i in i_arr:
    for j in j_arr:
        for k in k_arr:
            for l in l_arr:
                if((12 * ((2**i) * (3**j) * (5**k) * (11**l))) <= 3300):
                # if (not ((12 * (2**i * 3**j * 5**k)) in N_arr)):
                    # print(f"i = {i}, j = {j}, k = {k}")
                    N_arr.append(12 * (2**i * 3**j * 5**k * 11**l))

N_arr.sort()

print(max(N_arr))
print(len(N_arr))

print(N_arr)
print(i_arr)
print(j_arr)
print(k_arr)
print(l_arr)


3300
75
[-2036816896, -1594149888, -1472279296, -531383296, 12, 24, 36, 48, 60, 72, 96, 108, 120, 132, 144, 180, 192, 216, 240, 264, 288, 300, 324, 360, 384, 396, 432, 480, 528, 540, 576, 600, 648, 660, 720, 768, 792, 864, 900, 960, 972, 1056, 1080, 1152, 1188, 1200, 1296, 1320, 1440, 1452, 1500, 1536, 1584, 1620, 1728, 1800, 1920, 1944, 1980, 2112, 2160, 2304, 2376, 2400, 2592, 2640, 2700, 2880, 2904, 2916, 3000, 3072, 3168, 3240, 3300]
[0 1 2 3 4 5 6 7 8]
[0 1 2 3 4 5]
[0 1 2 3]
[0 1 2]


C:\Users\makotolagano\AppData\Local\Temp\ipykernel_12968\1146246817.py:11: RuntimeWarning: overflow encountered in scalar multiply
  if((12 * ((2**i) * (3**j) * (5**k) * (11**l))) <= 3300):
C:\Users\makotolagano\AppData\Local\Temp\ipykernel_12968\1146246817.py:14: RuntimeWarning: overflow encountered in scalar multiply
  N_arr.append(12 * (2**i * 3**j * 5**k * 11**l))
